# Train W1, W2, W3, W4 from One CSV (30% Label Noise)

This notebook uses **one input CSV only**:

`deterministic_compatibility_dataset_all_4_scores_with_30pct_noise.csv`

It uses only these five columns:
- `skin_compatibility`
- `face_compatibility`
- `body_compatibility`
- `design_level_score`
- `suitable_class`

All other columns are ignored.

## Workflow
1. Load the single CSV.
2. Select only the four scores and `suitable_class`.
3. Split rows into 80% training and 20% testing.
4. Train a Random Forest on the training labels as they appear in the CSV (including noise).
5. Evaluate against the test labels as they appear in the same CSV.
6. Calculate normalized W1–W4 using permutation importance on the test set.
7. Save and download the weights, metrics, audit, classification report, predictions, and trained model.

**Important:** Because the test labels also come from the 30%-noise file, the reported evaluation metrics and permutation-importance weights are measured against noisy test labels. This is intentional in this notebook, as requested. The weights are model-based relative importance shares, not coefficients from a linear weighted-sum formula.


In [1]:
# 1. Upload the ONE input CSV
from google.colab import files
uploaded = files.upload()

import os
import numpy as np
import pandas as pd

expected_name = "deterministic_compatibility_dataset_all_4_scores_with_30pct_noise.csv"
matches = [
    name for name in uploaded
    if os.path.basename(name).lower() == expected_name.lower()
]

if not matches:
    raise FileNotFoundError(
        f"Please upload only this file: {expected_name}\n"
        f"Uploaded files: {list(uploaded.keys())}"
    )

INPUT_FILE = matches[0]
raw_df = pd.read_csv(INPUT_FILE)

print("Input file:", os.path.basename(INPUT_FILE))
print("Original shape:", raw_df.shape)


ModuleNotFoundError: No module named 'google'

In [ ]:
# 2. Keep ONLY the four scores and class label
FEATURES = [
    "skin_compatibility",
    "face_compatibility",
    "body_compatibility",
    "design_level_score",
]
TARGET = "suitable_class"
REQUIRED_COLUMNS = FEATURES + [TARGET]

missing = [col for col in REQUIRED_COLUMNS if col not in raw_df.columns]
if missing:
    raise ValueError(f"Required columns missing from CSV: {missing}")

# Explicitly discard every other column.
data = raw_df[REQUIRED_COLUMNS].copy()

for col in FEATURES + [TARGET]:
    data[col] = pd.to_numeric(data[col], errors="coerce")

rows_before = len(data)
data = data.dropna(subset=REQUIRED_COLUMNS).reset_index(drop=True)
rows_dropped = rows_before - len(data)

# The four compatibility scores should be numeric and within [0, 1].
if not data[FEATURES].apply(lambda col: col.between(0, 1).all()).all():
    raise ValueError("One or more compatibility score columns contain values outside [0, 1].")

# Class labels are identifiers, so convert to integers.
if not np.allclose(data[TARGET], np.round(data[TARGET])):
    raise ValueError("suitable_class contains non-integer class IDs.")
data[TARGET] = data[TARGET].astype(int)

X = data[FEATURES].copy()
y = data[TARGET].copy()

print("Rows used:", len(data))
print("Rows dropped due to missing required values:", rows_dropped)
print("Features used:", FEATURES)
print("Target used:", TARGET)
print("Number of classes:", y.nunique())
print("Class-label distribution (top 10):")
display(y.value_counts().head(10).rename_axis("class").reset_index(name="rows"))
display(data.head())


In [ ]:
# 3. Split the one dataset into 80% train and 20% test
from sklearn.model_selection import train_test_split

SEED = 42

# Stratification preserves class proportions where class counts allow it.
class_counts = y.value_counts()
can_stratify = class_counts.min() >= 2
stratify_arg = y if can_stratify else None

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=SEED,
    stratify=stratify_arg
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("Training proportion:", f"{len(X_train) / len(data):.1%}")
print("Testing proportion:", f"{len(X_test) / len(data):.1%}")
print("Both splits use the class labels exactly as stored in the 30%-noise CSV.")


In [ ]:
# 4. Train Random Forest using the labels in the noisy CSV
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=300,
    max_features=None,
    min_samples_leaf=1,
    class_weight="balanced_subsample",
    random_state=SEED,
    n_jobs=-1
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)

print("Training completed.")
print("Model:", model.__class__.__name__)
print("Number of trees:", model.n_estimators)


In [ ]:
# 5. Calculate evaluation metrics against the held-out noisy labels
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    top_k_accuracy_score
)

test_accuracy = accuracy_score(y_test, y_pred)
balanced_accuracy = balanced_accuracy_score(y_test, y_pred)
macro_precision = precision_score(y_test, y_pred, average="macro", zero_division=0)
macro_recall = recall_score(y_test, y_pred, average="macro", zero_division=0)
macro_f1 = f1_score(y_test, y_pred, average="macro", zero_division=0)

if len(model.classes_) >= 3:
    top3_accuracy = top_k_accuracy_score(
        y_test, y_prob, k=3, labels=model.classes_
    )
else:
    top3_accuracy = np.nan

metrics_df = pd.DataFrame([{
    "model": "RandomForestClassifier",
    "total_rows_used": len(data),
    "train_rows": len(X_train),
    "test_rows": len(X_test),
    "split_ratio": "80:20",
    "label_noise_in_input_file": "30% (as specified)",
    "evaluation_labels": "Noisy labels from the same input CSV",
    "test_accuracy": test_accuracy,
    "balanced_test_accuracy": balanced_accuracy,
    "top_3_test_accuracy": top3_accuracy,
    "macro_precision": macro_precision,
    "macro_recall": macro_recall,
    "macro_f1": macro_f1,
    "number_of_classes": int(y.nunique())
}])

print(f"Test accuracy:          {test_accuracy:.4f} ({test_accuracy:.2%})")
print(f"Balanced test accuracy: {balanced_accuracy:.4f} ({balanced_accuracy:.2%})")
print(f"Top-3 test accuracy:    {top3_accuracy:.4f}" if not np.isnan(top3_accuracy) else "Top-3 test accuracy: N/A")
print(f"Macro precision:        {macro_precision:.4f}")
print(f"Macro recall:           {macro_recall:.4f}")
print(f"Macro F1-score:         {macro_f1:.4f}")

print("\nClassification report:")
print(classification_report(y_test, y_pred, zero_division=0))
display(metrics_df)


In [ ]:
# 6. Calculate W1, W2, W3, W4 using permutation importance
from sklearn.inspection import permutation_importance

perm = permutation_importance(
    model,
    X_test,
    y_test,
    scoring="accuracy",
    n_repeats=8,
    random_state=SEED,
    n_jobs=-1
)

# Negative mean importance values are clipped to zero before normalization.
importance = np.maximum(perm.importances_mean, 0)
importance_method = "permutation_importance_on_noisy_test_labels"

# Fallback if all permutation means are non-positive.
if importance.sum() == 0:
    importance = np.maximum(model.feature_importances_, 0)
    importance_method = "random_forest_impurity_importance_fallback"

if importance.sum() == 0:
    raise ValueError("All feature importances are zero; W1-W4 cannot be calculated.")

normalized_weights = importance / importance.sum()
weight_names = ["W1", "W2", "W3", "W4"]

weights_df = pd.DataFrame({
    "weight": weight_names,
    "feature": FEATURES,
    "permutation_importance_mean": perm.importances_mean,
    "permutation_importance_std": perm.importances_std,
    "normalized_weight": normalized_weights,
    "weight_percent": normalized_weights * 100,
    "importance_method": importance_method
})

W1, W2, W3, W4 = normalized_weights.tolist()

display(weights_df)
print(f"W1 (skin)   = {W1:.6f}  ({W1:.2%})")
print(f"W2 (face)   = {W2:.6f}  ({W2:.2%})")
print(f"W3 (body)   = {W3:.6f}  ({W3:.2%})")
print(f"W4 (design) = {W4:.6f}  ({W4:.2%})")
print(f"Sum of weights = {normalized_weights.sum():.8f}")


In [ ]:
# 7. Create classification report, split audit, and test predictions
report_df = pd.DataFrame(
    classification_report(
        y_test, y_pred, output_dict=True, zero_division=0
    )
).transpose().reset_index().rename(columns={"index": "class_or_average"})

# Audit contains only the four scores, class, and split label.
audit_df = data.copy()
audit_df["split"] = "test"
audit_df.loc[X_train.index, "split"] = "train"

# Predictions also use only the selected five input fields plus prediction fields.
predictions_df = X_test.copy()
predictions_df["actual_class_from_noisy_file"] = y_test
predictions_df["predicted_class"] = y_pred
predictions_df["prediction_correct_against_noisy_label"] = (
    y_test.to_numpy() == y_pred
)
predictions_df["top_3_predicted_classes"] = [
    ",".join(map(str, model.classes_[np.argsort(row)[-3:][::-1]]))
    for row in y_prob
]

display(report_df.head(10))
display(predictions_df.head())


In [ ]:
# 8. Save all output files
import joblib

weights_df.to_csv("global_weights_w1_w2_w3_w4.csv", index=False)
metrics_df.to_csv("model_evaluation_metrics.csv", index=False)
audit_df.to_csv("training_test_split_label_noise_audit.csv", index=False)
report_df.to_csv("classification_report.csv", index=False)
predictions_df.to_csv("test_predictions.csv", index=False)

joblib.dump({
    "model": model,
    "feature_columns": FEATURES,
    "target_column": TARGET,
    "weights": weights_df,
    "metrics": metrics_df
}, "compatibility_class_model.joblib")

output_files = [
    "global_weights_w1_w2_w3_w4.csv",
    "model_evaluation_metrics.csv",
    "training_test_split_label_noise_audit.csv",
    "classification_report.csv",
    "test_predictions.csv",
    "compatibility_class_model.joblib"
]

print("Created output files:")
for filename in output_files:
    print(" -", filename)


In [ ]:
# 9. Download all output files
for filename in output_files:
    files.download(filename)


## Output files

| Output file | Contents |
|---|---|
| `global_weights_w1_w2_w3_w4.csv` | W1–W4, raw feature importance, normalized weights and percentages |
| `model_evaluation_metrics.csv` | Accuracy, balanced accuracy, top-3 accuracy, macro precision, recall and F1 |
| `training_test_split_label_noise_audit.csv` | Only the four scores, class label and train/test split |
| `classification_report.csv` | Per-class precision, recall, F1-score and support |
| `test_predictions.csv` | Four scores, actual noisy class, predicted class and correctness |
| `compatibility_class_model.joblib` | Trained model, feature names, weights and metrics |

**The notebook uses one CSV only.** It ignores every input column except the four compatibility scores and `suitable_class`. Since the held-out test labels are also noisy, all evaluation metrics and permutation-importance weights are calculated against those noisy labels, as requested.
